In [4]:
from unsloth import FastLanguageModel
from datasets import Dataset

import faiss
from langchain.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pythainlp.tokenize import word_tokenize

import pandas as pd
import numpy as np

from tqdm import tqdm
import torch
import json
import sys
import os

ROOT_DIR = "/project/lt200304-dipmt/paweekorn"
MODEL_ID = "gemma3-4b-pt"

# import our custom function
sys.path.append(os.path.join(ROOT_DIR, "script"))
from utils.llm import init_model, train_model
from utils.retrieval import process_query

## Overview

In [2]:
train_df = pd.read_csv(f"{ROOT_DIR}/data/train_40k.csv")
test_df = pd.read_csv(f"{ROOT_DIR}/data/DS01/test_v1.csv")

with open(f"{ROOT_DIR}/data/wipo/WIPO.json", "r") as f:
    wipo_data = json.load(f)
    wipo_data = {int(k): v for k, v in wipo_data.items()}

train_df['WIPO'] = train_df['NAME'].map(wipo_data)
test_df['WIPO'] = test_df['NAME'].map(wipo_data)

print(train_df.shape)
train_df.head()

(39706, 4)


,ENG,THA,NAME,WIPO
0,"condiment, namely, pepper sauce","เครื่องปรุงรส, คือ, ซอสพริกไทย",30,"Coffee, tea, cocoa and substitutes therefor; r..."
1,machine and motor oil,น้ำมันเครื่องและมอเตอร์,4,Industrial oils and greases; lubricants; dust ...
2,medical analysis for diagnostic or treatment p...,วิเคราะห์ทางการแพทย์เพื่อการวินิจฉัยหรือการรักษา,44,Medical services; veterinary services; hygieni...
3,cleaning preparation for window pane,สารเตรียมขึ้นสำหรับทำความสะอาดบานกระจกหน้าต่าง,3,Non-medicated cosmetics and toiletry preparati...
4,protective padding for sport,แผ่นป้องกันสำหรับกีฬา,28,"Games, toys and playthings; video game apparat..."


In [3]:
model, tokenizer = init_model(model_path=f"{ROOT_DIR}/models/base/{MODEL_ID}",
                             max_seq_length=4096, load_in_4bit=True, rank=64)

==((====))==  Unsloth 2025.8.4: Fast Gemma3 patching. Transformers: 4.55.0. vLLM: 0.10.1.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.496 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


## Data Prep

In [10]:
retriever = "all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=f"{ROOT_DIR}/models/retriever/{retriever}")
vectorstore = FAISS.load_local(
    f"{ROOT_DIR}/vector/en2th/{retriever}", 
    embeddings,
    allow_dangerous_deserialization=True
)
gpu_index = faiss.index_cpu_to_gpu(faiss.StandardGpuResources(), 0, vectorstore.index)
vectorstore.index = gpu_index

In [11]:
with open(f"{ROOT_DIR}/data/prompt/base_en2th.txt") as f:
    instruction = f.read()

def formatting_prompt(df):
    batch = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        prompt = [
            { "role": "user", "content": instruction.format(
                WIPO=row['WIPO'], 
                RAG_DOC=process_query(
                    embeddings=embeddings, vectorstore=vectorstore, query=row['ENG']
                ), 
                ENGLISH=row['ENG']) }, 
            { "role": "assistant", "content": 
              f'''{{"thai_translation": "{row['THA']}" }}''' 
            }
        ]
        message = tokenizer.apply_chat_template(
            prompt, tokenize=False, add_generation_prompt=False,
        )
        batch.append({'text': message})

    return Dataset.from_list(batch)

train_set = formatting_prompt(train_df)
test_set = formatting_prompt(test_df)
print(train_set['text'][0])

 42%|████▏     | 16708/39706 [02:10<02:59, 128.30it/s]


KeyboardInterrupt: 

## Model Training

In [9]:
train_stats = train_model(output_dir=f"{ROOT_DIR}/models/adapter/{MODEL_ID}", epochs=2)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/39706 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2785 [00:00<?, ? examples/s]

[2025-11-16 04:40:20,177] [INFO] [real_accelerator.py:260:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/bin/../lib/gcc/x86_64-conda-linux-gnu/12.4.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/bin/../lib/gcc/x86_64-conda-linux-gnu/12.4.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-11-16 04:40:27,959] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 39,706 | Num Epochs = 1 | Total steps = 621
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 131,153,920 of 4,431,233,392 (2.96% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.530100,0.659574
200,0.314700,0.646952
300,0.300000,0.638213
400,0.287400,0.632406
500,0.282100,0.628884
600,0.281500,0.628926


Unsloth: Not an error, but Gemma3ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
